# HW08 — Почувствуй мощь трансформеров в бою

Задача бинарной классификации лингвистической приемлемости русских предложений на [RuCoLA](https://github.com/RussianNLP/RuCoLA). Метрики: **Matthews correlation coefficient (MCC)** — основная метрика RuCoLA — и accuracy.

Эксперименты: RuBERT fine-tuning, RuGPT3 zero/few-shot (0, 1, 2, 4 примера), RuT5 fine-tuning. Большие модели запускаются флагом `RUN_TRANSFORMERS=True`; по умолчанию ноутбук выполняет проверочный TF–IDF baseline и не выдаёт вымышленные transformer-метрики.

In [8]:
from pathlib import Path
import random, time
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_TRANSFORMERS = False   # На GPU поменять на True и выполнить Run All
FAST_MODE = False          # True использует компактные модели для отладки
EPOCHS = 3
BATCH_SIZE = 16
MAX_LENGTH = 128
DATA_DIR = Path('data'); RESULTS_DIR = Path('results'); RESULTS_DIR.mkdir(exist_ok=True)
print('Device:', DEVICE, '| transformers:', RUN_TRANSFORMERS)

Device: cpu | transformers: False


## 1. Данные и разбиение

In [9]:
train_full = pd.read_csv(DATA_DIR / 'in_domain_train.csv')
test = pd.read_csv(DATA_DIR / 'in_domain_dev.csv')
display(train_full.head())
print(train_full.shape, test.shape, train_full.columns.tolist())
print('Train labels:', train_full['acceptable'].value_counts(normalize=True).sort_index().round(3).to_dict())

train, val = train_test_split(
    train_full, test_size=0.20, random_state=SEED,
    stratify=train_full['acceptable'])
print('train/val/test:', len(train), len(val), len(test))

,id,sentence,acceptable,error_type,detailed_source
0,0,"Вдруг решетка беззвучно поехала в сторону, и н...",1,0,Paducheva2004
1,1,Этим летом не никуда ездили.,0,Syntax,Rusgram
2,2,Только Иван выразил какую бы то ни было готовн...,1,0,Paducheva2013
3,3,"Теперь ты видишь собственными глазами, как тут...",1,0,Paducheva2010
4,4,На поверку вся теория оказалась полной чепухой.,1,0,Paducheva2010


(7869, 5) (983, 5) ['id', 'sentence', 'acceptable', 'error_type', 'detailed_source']
Train labels: {0: 0.255, 1: 0.745}
train/val/test: 6295 1574 983


## 2. Проверочный baseline

Символьные n-граммы часто дают разумный baseline для грамматической приемлемости: они улавливают окончания и локальные конструкции. Он также проверяет, что разбиение и метрики работают до скачивания больших весов.

In [10]:
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, max_features=100_000)),
    ('model', LogisticRegression(max_iter=2000, C=3, class_weight='balanced', random_state=SEED))
])
baseline.fit(train['sentence'], train['acceptable'])
results = []
for split_name, frame in [('val', val), ('test', test)]:
    pred = baseline.predict(frame['sentence'])
    results.append({'model': 'TF-IDF baseline', 'split': split_name,
                    'MCC': matthews_corrcoef(frame['acceptable'], pred),
                    'accuracy': accuracy_score(frame['acceptable'], pred)})
display(pd.DataFrame(results).round(4))

,model,split,MCC,accuracy
0,TF-IDF baseline,val,0.1916,0.6658
1,TF-IDF baseline,test,0.2390,0.6907


## 3. RuBERT fine-tuning

Полная модель: `DeepPavlov/rubert-base-cased`. В `FAST_MODE` для проверки кода используется `cointegrated/rubert-tiny2`. Разбиение train/val фиксировано и стратифицировано; test применяется только к финальному checkpoint с лучшим validation MCC.

In [11]:
if RUN_TRANSFORMERS:
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    BERT_MODEL = 'cointegrated/rubert-tiny2' if FAST_MODE else 'DeepPavlov/rubert-base-cased'

    class ClassificationDataset(Dataset):
        def __init__(self, frame, tokenizer):
            self.sentences = frame['sentence'].tolist()
            self.labels = frame['acceptable'].astype(int).tolist()
            self.tokenizer = tokenizer
        def __len__(self): return len(self.labels)
        def __getitem__(self, index):
            item = self.tokenizer(self.sentences[index], truncation=True,
                                  max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
            return {k: v.squeeze(0) for k, v in item.items()} | {'labels': torch.tensor(self.labels[index])}

    def evaluate_classifier(model, loader):
        model.eval(); labels, predictions = [], []
        with torch.no_grad():
            for batch in loader:
                y = batch.pop('labels')
                logits = model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits.cpu()
                labels.extend(y.tolist()); predictions.extend(logits.argmax(1).tolist())
        return matthews_corrcoef(labels, predictions), accuracy_score(labels, predictions)

    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    bert = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2).to(DEVICE)
    loaders = {name: DataLoader(ClassificationDataset(frame, bert_tokenizer), batch_size=BATCH_SIZE,
                                shuffle=(name == 'train'))
               for name, frame in [('train', train), ('val', val), ('test', test)]}
    optimizer = torch.optim.AdamW(bert.parameters(), lr=2e-5, weight_decay=0.01)
    best_mcc, best_state = -1, None
    for epoch in range(EPOCHS):
        bert.train()
        for batch in loaders['train']:
            optimizer.zero_grad(set_to_none=True)
            loss = bert(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0); optimizer.step()
        val_mcc, val_acc = evaluate_classifier(bert, loaders['val'])
        print('RuBERT epoch', epoch + 1, 'val MCC', round(val_mcc, 4))
        if val_mcc > best_mcc:
            best_mcc = val_mcc
            best_state = {k: v.detach().cpu().clone() for k, v in bert.state_dict().items()}
    bert.load_state_dict(best_state)
    for split_name in ['val', 'test']:
        mcc, acc = evaluate_classifier(bert, loaders[split_name])
        results.append({'model': BERT_MODEL, 'split': split_name, 'MCC': mcc, 'accuracy': acc})
else:
    print('RuBERT пропущен: установите transformers sentencepiece и включите RUN_TRANSFORMERS.')

RuBERT пропущен: установите transformers sentencepiece и включите RUN_TRANSFORMERS.


## 4. RuGPT3 zero-/few-shot

Вместо нестабильного свободного generation сравнивается условная log-likelihood двух продолжений: `да` и `нет`. Перебираются три формулировки инструкции и 0/1/2/4 демонстрации. Это воспроизводимый zero/few-shot классификатор без обучения весов.

In [12]:
if RUN_TRANSFORMERS:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    GPT_MODEL = 'ai-forever/rugpt3small_based_on_gpt2' if FAST_MODE else 'ai-forever/rugpt3large_based_on_gpt2'
    gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL)
    gpt = AutoModelForCausalLM.from_pretrained(GPT_MODEL).to(DEVICE).eval()
    if gpt_tokenizer.pad_token is None: gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

    templates = [
        'Грамматически ли корректно русское предложение? Ответ: да или нет.\n',
        'Определи лингвистическую приемлемость. Допустимые ответы: да, нет.\n',
        'Задача: проверить правильность предложения на русском языке.\n']
    demonstrations = pd.concat([train[train.acceptable == 1].head(2),
                                train[train.acceptable == 0].head(2)]).sample(frac=1, random_state=SEED)

    def build_prompt(sentence, template, shots):
        demos = ''.join(f'Предложение: {r.sentence}\nОтвет: {"да" if r.acceptable else "нет"}\n\n'
                        for _, r in demonstrations.head(shots).iterrows())
        return template + demos + f'Предложение: {sentence}\nОтвет:'

    def completion_nll(prompt, completion):
        prompt_ids = gpt_tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
        full = gpt_tokenizer(prompt + ' ' + completion, return_tensors='pt').input_ids.to(DEVICE)
        labels = full.clone(); labels[:, :prompt_ids.shape[1]] = -100
        with torch.no_grad(): return gpt(full, labels=labels).loss.item()

    # Ограничение делает полный перебор практичным; размер явно фиксируется в результатах.
    gpt_test = test.sample(min(200, len(test)), random_state=SEED)
    for template_id, template in enumerate(templates, 1):
        for shots in [0, 1, 2, 4]:
            pred = [int(completion_nll(build_prompt(s, template, shots), 'да') <
                        completion_nll(build_prompt(s, template, shots), 'нет'))
                    for s in gpt_test['sentence']]
            results.append({'model': f'RuGPT3 prompt={template_id}, shots={shots}', 'split': 'test-200',
                            'MCC': matthews_corrcoef(gpt_test.acceptable, pred),
                            'accuracy': accuracy_score(gpt_test.acceptable, pred)})
else:
    print('RuGPT3 пропущен: RUN_TRANSFORMERS=False')

RuGPT3 пропущен: RUN_TRANSFORMERS=False


## 5. RuT5 fine-tuning

Классификация задаётся text-to-text: вход `приемлемость: <sentence>`, цель `да`/`нет`. Валидация и выбор checkpoint выполняются по MCC.

In [13]:
if RUN_TRANSFORMERS:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    T5_MODEL = 'cointegrated/rut5-small' if FAST_MODE else 'ai-forever/ruT5-base'
    t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
    t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL).to(DEVICE)

    class T5Dataset(torch.utils.data.Dataset):
        def __init__(self, frame): self.frame = frame.reset_index(drop=True)
        def __len__(self): return len(self.frame)
        def __getitem__(self, i):
            row = self.frame.iloc[i]
            source = t5_tokenizer('приемлемость: ' + row.sentence, truncation=True,
                                  max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
            target = t5_tokenizer('да' if row.acceptable else 'нет', max_length=4,
                                  padding='max_length', return_tensors='pt')
            labels = target.input_ids.squeeze(0); labels[labels == t5_tokenizer.pad_token_id] = -100
            return {k: v.squeeze(0) for k, v in source.items()} | {'labels': labels}

    t5_loaders = {name: torch.utils.data.DataLoader(T5Dataset(frame), batch_size=BATCH_SIZE,
                                                   shuffle=(name == 'train'))
                  for name, frame in [('train', train), ('val', val), ('test', test)]}
    def evaluate_t5(loader):
        true, pred = [], []
        t5.eval()
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop('labels')
                output = t5.generate(**{k: v.to(DEVICE) for k, v in batch.items()}, max_new_tokens=3)
                texts = t5_tokenizer.batch_decode(output, skip_special_tokens=True)
                pred.extend([int(x.strip().lower().startswith('да')) for x in texts])
                restored = labels.clone(); restored[restored == -100] = t5_tokenizer.pad_token_id
                true_texts = t5_tokenizer.batch_decode(restored, skip_special_tokens=True)
                true.extend([int(x.strip().lower().startswith('да')) for x in true_texts])
        return matthews_corrcoef(true, pred), accuracy_score(true, pred)

    t5_optimizer = torch.optim.AdamW(t5.parameters(), lr=1e-4, weight_decay=0.01)
    best_mcc, best_state = -1, None
    for epoch in range(EPOCHS):
        t5.train()
        for batch in t5_loaders['train']:
            t5_optimizer.zero_grad(set_to_none=True)
            loss = t5(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(t5.parameters(), 1.0); t5_optimizer.step()
        val_mcc, _ = evaluate_t5(t5_loaders['val'])
        print('RuT5 epoch', epoch + 1, 'val MCC', round(val_mcc, 4))
        if val_mcc > best_mcc:
            best_mcc = val_mcc
            best_state = {k: v.detach().cpu().clone() for k, v in t5.state_dict().items()}
    t5.load_state_dict(best_state)
    for split_name in ['val', 'test']:
        mcc, acc = evaluate_t5(t5_loaders[split_name])
        results.append({'model': T5_MODEL, 'split': split_name, 'MCC': mcc, 'accuracy': acc})
else:
    print('RuT5 пропущен: RUN_TRANSFORMERS=False')

RuT5 пропущен: RUN_TRANSFORMERS=False


## 6. Сравнение

In [14]:
comparison = pd.DataFrame(results).sort_values(['split', 'MCC'], ascending=[True, False])
display(comparison.round(4))
comparison.to_csv(RESULTS_DIR / 'comparison.csv', index=False)
if not RUN_TRANSFORMERS:
    print('В таблице пока только baseline. Для сдачи transformer-части нужен GPU-запуск с RUN_TRANSFORMERS=True.')

,model,split,MCC,accuracy
1,TF-IDF baseline,test,0.2390,0.6907
0,TF-IDF baseline,val,0.1916,0.6658


В таблице пока только baseline. Для сдачи transformer-части нужен GPU-запуск с RUN_TRANSFORMERS=True.


## Выводы

Сравнивать методы следует прежде всего по test MCC. Fine-tuning использует веса для решения конкретной задачи и обычно должен быть сильнее zero/few-shot. RuGPT3 не обучается и чувствителен к формулировке затравки и числу демонстраций — именно поэтому перебираются 12 конфигураций. RuT5 формулирует классификацию как генерацию короткой метки. Финальные численные выводы следует записывать только после GPU-запуска всех блоков; ноутбук намеренно не подставляет фиктивные результаты.